# BVC Terminal — Collecte historique via BVCscrap

Ce notebook collecte 90 jours d'OHLCV pour **72/77 tickers** BVC via BVCscrap (Médias24),
calcule les indicateurs techniques (RSI, MA20/50, MACD, Bollinger, Stoch),
et sauvegarde `historical_data.json` pour le terminal.

**Exécuter depuis Google Colab** (IP non bloquée par Médias24).

Ensuite : committer et pousser `pipeline/historical_data.json` sur le repo GitHub.

In [ ]:
# Cloner le repo (remplacer par votre token si privé)
import os
REPO = 'abdmoutalib207-lang/-bvc-analyzer'
TOKEN = ''  # Laisser vide si public, ou mettre votre GitHub token

if TOKEN:
    !git clone https://{TOKEN}@github.com/{REPO} bvc-analyzer
else:
    !git clone https://github.com/{REPO} bvc-analyzer

os.chdir('bvc-analyzer')
!pip install bvcscrap pandas numpy -q

In [ ]:
# Lancer la collecte complète (72 tickers, ~5 minutes)
!python pipeline/collect_history_bvcscrap.py --days 95

In [ ]:
# Vérification du résultat
import json
with open('pipeline/historical_data.json') as f:
    data = json.load(f)

tickers = {k: v for k, v in data.items() if not k.startswith('_')}
print(f"Tickers collectés : {len(tickers)}")
print(f"Mis à jour : {data.get('_updated', 'N/A')}")
print()

# Afficher les tickers avec MA20 ≈ MA50 (still ~stat)
stat_list = [t for t, v in tickers.items() if abs(v.get('ma20',0) - v.get('ma50',0)) < 0.1]
print(f"Tickers ~stat restants : {len(stat_list)} → {stat_list}")
print()

# Aperçu
import pandas as pd
rows = []
for t, v in sorted(tickers.items()):
    rows.append({'Ticker': t, 'RSI': v.get('rsi'), 'MA20': v.get('ma20'),
                 'MA50': v.get('ma50'), 'H90': v.get('h90'), 'L90': v.get('l90'),
                 'Bougies': v.get('n_candles')})
pd.DataFrame(rows).set_index('Ticker')

In [ ]:
# Committer et pousser historical_data.json
!git config user.email 'bvc-colab@users.noreply.github.com'
!git config user.name 'BVC Colab'

from datetime import datetime
date_str = datetime.now().strftime('%Y-%m-%d')

!git add pipeline/historical_data.json
!git commit -m "data: historical_data.json BVCscrap {date_str}"

# Push (nécessite TOKEN si repo privé ou HTTPS)
if TOKEN:
    !git push https://{TOKEN}@github.com/{REPO} main
else:
    !git push origin main
    
print('✓ historical_data.json poussé sur GitHub')